# 📖 Notebook 2: Alerting Rules & Thresholds

Dashboards are great for watching metrics, but humans can't stare at screens 24/7. **Alerts** automatically detect when something is wrong and notify the right people.

## Learning Objectives

By the end of this notebook, you'll understand:
- How alert rules work (query + threshold + duration)
- The polling-based evaluation loop (how Prometheus does it)
- Alert states: inactive → pending → firing → resolved
- Notification routing: grouping, deduplication, and silencing
- How to store and manage alert rules in a database

## 🛠️ Setup

Make sure your infrastructure is running:

```bash
cd 06-system-designs/metrics-monitoring
docker compose up -d
```

**Important**: Run Notebook 1 first! The metrics server from Notebook 1 must be running so Prometheus has data to evaluate alerts against.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import requests
import time
import json
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "metrics_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

PROMETHEUS_URL = "http://localhost:9090"

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

def prom_query(query: str) -> list:
    resp = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={"query": query})
    data = resp.json()
    if data["status"] != "success":
        return []
    return data["data"]["result"]

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

try:
    resp = requests.get(f"{PROMETHEUS_URL}/-/healthy", timeout=5)
    print("✅ Connected to Prometheus")
except Exception as e:
    print(f"❌ Prometheus failed: {e}")

## 🤔 What Is an Alert Rule?

An alert rule is a **condition** that, when true for long enough, triggers a notification.

Every alert rule has three parts:

```
┌─────────────────────────────────────────────────────┐
│  Alert Rule: "High CPU"                             │
│                                                     │
│  Query:     avg(cpu_usage{region="us-east"})        │
│  Threshold: > 90%                                   │
│  Duration:  for 5 minutes                           │
│                                                     │
│  Translation: "Alert me if the average CPU in       │
│  us-east stays above 90% for 5 minutes straight."   │
└─────────────────────────────────────────────────────┘
```

The **duration** is important! You don't want to get paged for a 2-second CPU spike that resolves itself. The "for" clause prevents false positives.

### Alert States

```
inactive ──(condition true)──► pending ──(still true for 5m)──► FIRING
    ▲                              │                              │
    └──────(condition false)────────┘   (condition false) ────► resolved
```

- **Inactive**: Condition is false — everything is fine
- **Pending**: Condition just became true — waiting for duration to expire
- **Firing**: Condition has been true for the full duration — send notifications!
- **Resolved**: Was firing, now the condition is false again — send "resolved" notification

### What `for:` Actually Means (the part people get wrong)

`for: 5m` does **not** mean "the average over the last 5 minutes breached". It means the
expression must evaluate to true at **every single evaluation** during those 5 minutes.
One false evaluation — a single scrape that dipped back under the threshold, or a scrape that
failed and returned no data — resets the alert straight to `inactive` and the clock starts
over from zero.

Two consequences worth internalising:

- A flapping metric can breach a threshold 90% of the time and still never fire a
  `for: 5m` alert. If you want "mostly bad for 5 minutes", put the window in the *expression*
  (`avg_over_time(...[5m]) > x`), not in `for:`.
- Detection is never faster than `for` + one evaluation interval. With
  `evaluation_interval: 15s` and `for: 1m`, the earliest you can page is ~60-75s after the
  problem starts.


In [ ]:
# Let's look at the alert rules stored in our Postgres database

conn = get_db_connection()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT name, query, threshold, operator, duration_seconds, severity, enabled
    FROM alert_rules
    ORDER BY severity DESC, name
""")
rules = cursor.fetchall()

print("📋 Alert Rules in Database")
print("=" * 80)
print(f"{'Name':<22} {'Condition':<35} {'For':>6} {'Severity':<10} {'On?'}")
print("-" * 80)
for rule in rules:
    condition = f"{rule['query']} {rule['operator']} {rule['threshold']}"
    duration = f"{rule['duration_seconds']}s"
    enabled = "✅" if rule['enabled'] else "❌"
    print(f"{rule['name']:<22} {condition:<35} {duration:>6} {rule['severity']:<10} {enabled}")

print()
print(f"Total rules: {len(rules)}")
print()
print("💡 These rules are stored in Postgres because they change infrequently.")
print("   The alert evaluator reads them periodically and runs the queries.")

conn.close()

In [ ]:
# Let's also see the notification channels and which alerts go where

conn = get_db_connection()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT
        ar.name AS alert_name,
        ar.severity,
        nc.name AS channel_name,
        nc.channel_type
    FROM alert_rules ar
    JOIN alert_notifications an ON ar.id = an.alert_rule_id
    JOIN notification_channels nc ON an.channel_id = nc.id
    ORDER BY ar.severity DESC, ar.name
""")
results = cursor.fetchall()

print("📣 Alert → Notification Routing")
print("=" * 65)
print(f"{'Alert':<22} {'Severity':<10} {'Channel':<18} {'Type'}")
print("-" * 65)
for row in results:
    print(f"{row['alert_name']:<22} {row['severity']:<10} {row['channel_name']:<18} {row['channel_type']}")

print()
print("💡 Critical alerts go to Slack AND PagerDuty (wake someone up!).")
print("   Warning alerts only go to Slack (can wait until morning).")

conn.close()

## 🔄 The Alert Evaluation Loop

The alert evaluator is a simple **polling loop** that runs on a schedule:

```
Every 60 seconds:
  1. Load all enabled alert rules from Postgres
  2. For each rule, run its PromQL query against Prometheus
  3. Compare the result to the threshold
  4. Track how long the condition has been true (state management)
  5. If firing long enough → emit alert event
```

This is exactly how **Prometheus Alertmanager** works. The simplicity is the feature:  
alerts are just **scheduled queries**. Easy to reason about, easy to debug.

### Why Polling and Not Streaming?

- Our requirement says alerts should fire within **1 minute** — polling every 30-60 seconds is fine
- Polling reuses the same query engine as dashboards (no separate system)
- Stream processing (Flink/Kafka) adds complexity for marginal latency improvement
- Most production Prometheus setups use polling with 15-60 second intervals

In [ ]:
# Build a simple alert evaluator that polls Prometheus

# This maps alert rule IDs to their current state
alert_states = {}  # rule_id -> {"state": "inactive"|"pending"|"firing", "since": timestamp}

def evaluate_alert_rule(rule: dict, value_override: float | None = None) -> dict:
    """
    Evaluate a single alert rule against Prometheus.
    Returns the current evaluation result.

    `value_override` lets us feed the evaluator a scripted value instead of a live
    one. Real evaluators do the same thing in their unit tests — the state machine
    is the interesting part, and you cannot test it if you have to wait for
    production to misbehave.
    """
    # Map our rule's query to a PromQL query that Prometheus understands.
    # In production, this would be the actual PromQL expression.
    # Our demo metrics use 'demo_' prefix.
    query_map = {
        "avg(cpu_usage)": "avg(demo_cpu_usage)",
        "avg(memory_usage)": "avg(demo_memory_usage)",
        "avg(error_rate)": "avg(demo_error_rate)",
        "max(disk_usage)": "max(demo_cpu_usage)",  # reuse CPU as proxy
        "avg(http_request_duration_seconds)": "avg(demo_request_latency)",
        "max(queue_depth)": "max(demo_request_latency) * 500",
    }

    prom_expr = query_map.get(rule["query"], rule["query"])

    if value_override is not None:
        value = float(value_override)
    else:
        results = prom_query(prom_expr)
        if not results:
            return {"rule": rule["name"], "value": None, "breached": False, "error": "no data"}
        value = float(results[0]["value"][1])

    threshold = float(rule["threshold"])
    op = rule["operator"]

    # Evaluate the threshold condition
    breached = False
    if op == ">" and value > threshold:
        breached = True
    elif op == "<" and value < threshold:
        breached = True
    elif op == ">=" and value >= threshold:
        breached = True
    elif op == "<=" and value <= threshold:
        breached = True
    elif op == "==" and value == threshold:
        breached = True

    return {
        "rule": rule["name"],
        "query": prom_expr,
        "value": round(value, 2),
        "threshold": threshold,
        "operator": op,
        "breached": breached
    }

# Evaluate all rules once
conn = get_db_connection()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cursor.execute("SELECT * FROM alert_rules WHERE enabled = TRUE")
rules = cursor.fetchall()
conn.close()

print("🔍 Alert Evaluation (one-time check)")
print("=" * 75)
print(f"{'Rule':<22} {'Value':>8} {'Op':>3} {'Threshold':>10} {'Status'}")
print("-" * 75)

for rule in rules:
    result = evaluate_alert_rule(rule)
    status = "🔴 BREACHED" if result["breached"] else "🟢 OK"
    val = f"{result['value']:.2f}" if result['value'] is not None else "N/A"
    print(f"{result['rule']:<22} {val:>8} {result.get('operator', ''):>3} {result.get('threshold', ''):>10} {status}")

print()
print("💡 A single evaluation tells us the CURRENT state.")
print("   But alerts need sustained violations — that's what the 'for' duration does.")


In [ ]:
# Now let's build the FULL evaluation loop with state tracking.
# This tracks how long each alert has been breached.

r = get_redis_client()

def get_alert_state(rule_id: int) -> dict:
    """Get alert state from Redis (persistent across evaluator restarts)."""
    state = r.hgetall(f"alert_state:{rule_id}")
    if not state:
        return {"state": "inactive", "since": "0"}
    return state

def set_alert_state(rule_id: int, state: str, now: float):
    """Update alert state in Redis, stamped with the EVALUATION time."""
    r.hset(f"alert_state:{rule_id}", mapping={
        "state": state,
        "since": str(now)
    })

def evaluate_with_state(rule: dict, value_override: float | None = None,
                        now: float | None = None) -> dict:
    """
    Evaluate an alert rule WITH state tracking.
    Manages the inactive → pending → firing → resolved lifecycle.

    `now` is the evaluation timestamp, injected rather than read from the wall
    clock. That is not a testing hack — Prometheus does the same thing, passing
    the evaluation timestamp down into every rule so that a rule group evaluates
    as of one consistent instant even if the evaluation itself takes a while.
    It is also what lets us drive a full lifecycle below without sleeping.
    """
    rule_id = rule["id"]
    result = evaluate_alert_rule(rule, value_override=value_override)
    current_state = get_alert_state(rule_id)
    now = time.time() if now is None else now
    old_state = current_state["state"]
    since = float(current_state["since"])

    if result["breached"]:
        if old_state == "inactive":
            # Just started breaching → move to pending
            set_alert_state(rule_id, "pending", now)
            result["new_state"] = "pending"
        elif old_state == "pending":
            # Still breaching — check if duration exceeded
            elapsed = now - since
            if elapsed >= rule["duration_seconds"]:
                set_alert_state(rule_id, "firing", now)
                result["new_state"] = "firing"
                result["action"] = "🚨 SEND NOTIFICATION!"
            else:
                result["new_state"] = "pending"
                result["waiting"] = f"{elapsed:.0f}s / {rule['duration_seconds']}s"
        else:  # already firing
            result["new_state"] = "firing"
    else:
        if old_state in ("pending", "firing"):
            set_alert_state(rule_id, "inactive", now)
            result["new_state"] = "resolved" if old_state == "firing" else "inactive"
            if old_state == "firing":
                result["action"] = "✅ SEND RESOLVED NOTIFICATION"
        else:
            result["new_state"] = "inactive"

    result["old_state"] = old_state
    return result


# ---------------------------------------------------------------------------
# Watching the machine complete a lap.
#
# Two things make this deterministic, and both are deliberate:
#   * the VALUES are scripted, because waiting for the simulator to breach a
#     threshold on its own would take minutes; and
#   * the CLOCK is scripted, because the transitions depend on elapsed time, and
#     a demo that sleeps for real would produce a different answer on a loaded
#     machine than on an idle one. There is nothing to observe here that real
#     time can tell us and a virtual clock cannot.
# ---------------------------------------------------------------------------

DEMO_RULE = {
    "id": 9001,
    "name": "Scripted High CPU",
    "query": "avg(cpu_usage)",
    "threshold": 90.0,
    "operator": ">",
    "duration_seconds": 60,   # same `for: 1m` the real rules in alert_rules.yml use
}

STATE_ICON = {"inactive": "⚪", "pending": "🟡", "firing": "🔴", "resolved": "🟢"}


def run_script(rule: dict, script: list, title: str) -> list:
    """Evaluate `rule` once per (timestamp, value) pair. Returns states observed."""
    r.delete(f"alert_state:{rule['id']}")
    print(title)
    print(f"   {rule['query']} > {rule['threshold']}  for {rule['duration_seconds']}s"
          f"   (evaluated every 15s, as Prometheus does)")
    print()
    observed = []
    for ts, value in script:
        result = evaluate_with_state(rule, value_override=value, now=ts)
        observed.append(result["new_state"])
        extra = ""
        if "action" in result:
            extra = f"  → {result['action']}"
        elif "waiting" in result:
            extra = f"  (breached for {result['waiting']})"
        print(f"  t={ts:>4.0f}s  {STATE_ICON.get(result['new_state'], '❓')} "
              f"value={value:>6.1f}  {result['old_state']:<9}→ "
              f"{result['new_state']:<9}{extra}")
    print()
    return observed


# Case 1: a sustained breach. The condition holds at EVERY evaluation, so the
# `for` timer runs to completion and the alert fires.
sustained = [
    (  0.0, 72.4),   # calm
    ( 15.0, 94.1),   # breach begins — timer starts here
    ( 30.0, 95.6),   # 15s of breach, need 60
    ( 45.0, 93.8),   # 30s of breach
    ( 60.0, 94.9),   # 45s of breach
    ( 75.0, 96.2),   # 60s of breach → FIRING
    ( 90.0, 97.0),   # still bad → stays firing, no second page
    (105.0, 61.4),   # recovered → resolved
]
observed = run_script(DEMO_RULE, sustained, "🔄 Case 1 — sustained breach")

# Case 2: the same peak values, but ONE evaluation dips back under the threshold.
# This is the `for:` semantics people get wrong: the timer does not "mostly" run,
# it resets to zero. The alert never fires, even though the metric was over the
# line at 5 of the 7 evaluations.
flapping = [
    (  0.0, 72.4),
    ( 15.0, 94.1),   # breach begins — timer starts at t=15
    ( 30.0, 95.6),
    ( 45.0, 88.0),   # ONE evaluation under the line → back to inactive, timer lost
    ( 60.0, 94.5),   # breach begins again — timer restarts at t=60
    ( 75.0, 96.1),
    ( 90.0, 95.0),   # only 30s of breach so far — still pending when we stop looking
]
observed_flapping = run_script(DEMO_RULE, flapping, "🔄 Case 2 — flapping breach")

print("💡 The 'for' duration prevents flapping — brief spikes don't trigger alerts.")
print("   Only sustained violations (pending → firing) send notifications.")
print("   Case 2 is the trap: the metric was over the threshold at 5 of 7")
print("   evaluations and still never paged anyone. If you want 'mostly bad for")
print("   a minute', put the window in the EXPRESSION (avg_over_time(...[1m]))")
print("   rather than in `for:`.")
print()
print("   Note too that state lives in Redis, so an evaluator restart does NOT")
print("   reset the timer and re-page you for an incident already known.")

# These encode the lesson, and with an injected clock they are exact rather than
# approximate: the same inputs must always produce the same state sequence.
assert observed == ["inactive", "pending", "pending", "pending", "pending",
                    "firing", "firing", "resolved"], (
    f"sustained breach must walk inactive → pending → firing → resolved, got {observed}")
assert observed_flapping == ["inactive", "pending", "pending", "inactive",
                             "pending", "pending", "pending"], (
    f"one clean evaluation at t=45 must reset the alert to inactive and restart "
    f"the timer from t=60, so it is still only 30s into a 60s `for` at t=90; "
    f"got {observed_flapping}")
assert "firing" not in observed_flapping, (
    f"a breach interrupted by one clean evaluation must NEVER fire, got {observed_flapping}")


## 📋 Prometheus Alert Rules (Real Configuration)

Our Docker setup includes real Prometheus alert rules. Let's check their status!

**What you should expect to see.** The Notebook 1 simulator is tuned so that some of these
rules genuinely fire and others genuinely don't — a monitoring lab where every rule fires is
as useless as one where none do:

| Rule | Fires? | Why |
|---|---|---|
| `HighCpuUsage` (>80 for 1m) | ✅ about once per ~10.5-min cycle | the CPU sine wave crests above 80% for ~95s |
| `HighMemoryUsage` (>85 for 1m) | ✅ about once per ~12-min cycle | the memory sawtooth spends ~70s per cycle above 85% |
| `CriticalCpuUsage` (>95 for 30s) | ⚠️ rarely | only noise pushes past 95, never for two evaluations running |
| `HighErrorRate` (>5 for 30s) | ⚠️ rarely | error spikes are independent 3% events, so they almost never repeat |
| `HighLatency` (>2s for 1m) | ❌ effectively never | it watches the **average**, which sits near 0.45s even while the p99 is ~4s |

That last row is not a bug in the lab — it is the lesson. An alert on average latency is an
alert that will not wake you up while your slowest 1% of users are timing out. Notebook 3
computes the p99 from the histogram and shows the gap directly.

If a rule sits in `inactive` when you look, give it a couple of minutes and refresh — the
simulator's cycle is ~10 minutes long.


In [ ]:
# Query Prometheus for the status of its alert rules

resp = requests.get(f"{PROMETHEUS_URL}/api/v1/rules")
data = resp.json()

if data["status"] == "success":
    groups = data["data"]["groups"]
    print("📋 Prometheus Alert Rules (from alert_rules.yml)")
    print("=" * 70)

    for group in groups:
        print(f"\nGroup: {group['name']} (interval: {group['interval']}s)")
        print("-" * 70)

        for rule in group.get("rules", []):
            name = rule.get("name", "unknown")
            state = rule.get("state", "unknown")
            severity = rule.get("labels", {}).get("severity", "?")
            expr = rule.get("query", "")

            state_icon = {"inactive": "⚪", "pending": "🟡", "firing": "🔴"}
            icon = state_icon.get(state, "❓")

            print(f"  {icon} {name:<25} severity={severity:<10} state={state}")
            print(f"     expr: {expr}")

    print()
    print("💡 These rules are evaluated by Prometheus every 15 seconds.")
    print("   Open http://localhost:9090/alerts to see them in the Prometheus UI!")
else:
    print(f"❌ Failed to fetch rules: {data}")

## 🔔 Notification Routing

When an alert fires, we don't just send it everywhere. A **Notification Service** handles:

| Feature | What It Does | Why It Matters |
|---------|-------------|----------------|
| **Deduplication** | Don't re-send if alert is already firing | Prevents pager fatigue |
| **Grouping** | Combine 100 host alerts into 1 notification | One incident, one page |
| **Silencing** | Mute alerts during maintenance windows | No false alarms |
| **Escalation** | Re-notify via different channel if unacknowledged | Someone must respond |

This is exactly what **Prometheus Alertmanager** does. The key insight:  
**"Evaluating conditions" and "managing notifications" are separate problems.**

In [ ]:
# Simulate a notification service with deduplication and grouping

r = get_redis_client()

class NotificationService:
    """Simulates alert notification with dedup, grouping, and silencing."""

    def __init__(self, redis_client):
        self.r = redis_client
        self.sent_log = []  # log of notifications sent

    def process_alert_event(self, alert_name: str, state: str,
                           labels: dict, severity: str):
        """
        Process an alert event with dedup and grouping.
        Only sends notification on state transitions (new firing or resolved).
        """
        dedup_key = f"alert_dedup:{alert_name}"
        previous_state = self.r.get(dedup_key)

        # Deduplication: only notify on state TRANSITIONS
        if previous_state == state:
            return {"action": "deduplicated", "reason": f"Already in {state} state"}

        # Check for silencing
        silence_key = f"alert_silence:{alert_name}"
        if self.r.exists(silence_key):
            return {"action": "silenced", "reason": "Alert is silenced"}

        # Update state for future dedup checks
        self.r.setex(dedup_key, 3600, state)  # 1 hour TTL

        # Send notification
        notification = {
            "alert": alert_name,
            "state": state,
            "severity": severity,
            "labels": labels,
            "time": datetime.now().isoformat()
        }
        self.sent_log.append(notification)

        return {"action": "sent", "notification": notification}

    def silence_alert(self, alert_name: str, duration_minutes: int = 60):
        """Silence an alert for a specified duration."""
        self.r.setex(f"alert_silence:{alert_name}", duration_minutes * 60, "silenced")
        return f"Silenced {alert_name} for {duration_minutes} minutes"

# Demo the notification service.
# Clear the demo's Redis keys first so a re-run reproduces exactly the same four
# outcomes instead of tripping over state left behind by the previous run.
for _k in ("alert_dedup:HighCPU", "alert_dedup:HighMemory",
           "alert_silence:HighCPU", "alert_silence:HighMemory"):
    r.delete(_k)

ns = NotificationService(r)

print("📣 Notification Service Demo")
print("=" * 60)

# Scenario 1: New alert fires → should send
result = ns.process_alert_event("HighCPU", "firing",
    {"host": "web-1", "region": "us-east"}, "critical")
print(f"1. HighCPU fires:        {result['action']}")

# Scenario 2: Same alert fires again → should dedup
result = ns.process_alert_event("HighCPU", "firing",
    {"host": "web-1", "region": "us-east"}, "critical")
print(f"2. HighCPU fires again:  {result['action']} ({result['reason']})")

# Scenario 3: Alert resolves → should send
result = ns.process_alert_event("HighCPU", "resolved",
    {"host": "web-1", "region": "us-east"}, "critical")
print(f"3. HighCPU resolves:     {result['action']}")

# Scenario 4: Silence an alert, then try to fire it
ns.silence_alert("HighMemory", duration_minutes=30)
result = ns.process_alert_event("HighMemory", "firing",
    {"host": "web-2"}, "warning")
print(f"4. HighMemory (silenced): {result['action']} ({result['reason']})")

print()
print(f"Total notifications actually sent: {len(ns.sent_log)}")
print()
print("💡 Without dedup, the evaluator would page you every 60 seconds")
print("   for the same ongoing incident. Dedup = one page per incident.")

# Four events in, two notifications out. If dedup or silencing silently stops
# working, this fails instead of quietly paging someone four times.
assert len(ns.sent_log) == 2, (
    f"expected exactly 2 notifications (firing + resolved), got {len(ns.sent_log)}: "
    f"{[n['alert'] + ':' + n['state'] for n in ns.sent_log]}")
assert [n["state"] for n in ns.sent_log] == ["firing", "resolved"], (
    f"wrong notifications escaped: {ns.sent_log}")


In [ ]:
# Demonstrate GROUPING: combine related alerts into one notification

# Severity is an ORDERED enum, not a string. `max()` on the raw strings would
# sort alphabetically and rank "warning" above "critical" — so a group holding
# five critical alerts would page you as a warning. Rank it explicitly.
SEVERITY_RANK = {"info": 0, "warning": 1, "critical": 2}


def group_alerts(alert_events: list, group_by: str = "region",
                 window_seconds: int = 30) -> list:
    """
    Group alerts by a label within a time window.
    Instead of 100 individual host alerts, send 1 grouped notification.

    A grouped notification inherits the HIGHEST severity in the group: collapsing
    pages must never quietly downgrade one.
    """
    groups = {}
    for event in alert_events:
        key = event["labels"].get(group_by, "unknown")
        if key not in groups:
            groups[key] = []
        groups[key].append(event)

    grouped_notifications = []
    for group_key, events in groups.items():
        worst = max(events, key=lambda e: SEVERITY_RANK.get(e["severity"], -1))
        alert_name = events[0]["alert"]
        grouped_notifications.append({
            "group": f"{group_by}={group_key}",
            "alert_count": len(events),
            "severity": worst["severity"],
            "critical_count": sum(1 for e in events if e["severity"] == "critical"),
            "summary": f"{len(events)} hosts firing {alert_name} in {group_key}"
        })

    return grouped_notifications

# Simulate 20 servers all breaching CPU at once
events = []
for i in range(20):
    region = "us-east" if i < 12 else "us-west"
    events.append({
        "alert": "HighCPU",
        "labels": {"host": f"server-{i}", "region": region},
        "severity": "critical" if i < 5 else "warning"
    })

print("📣 Alert Grouping Demo")
print("=" * 60)
print(f"\nWithout grouping: {len(events)} individual notifications 😱")

grouped = group_alerts(events, group_by="region")
print(f"With grouping:    {len(grouped)} grouped notifications 👍")
print()
for g in grouped:
    print(f"  📨 [{g['severity']}] {g['summary']}")
    print(f"     ({g['alert_count']} alerts combined into 1 notification, "
          f"{g['critical_count']} of them critical)")

print()
print("💡 Grouping prevents your on-call engineer from getting 20 pages")
print("   for what is clearly ONE incident affecting a region.")
print("   But it must never LOSE information: us-east holds 5 critical alerts,")
print("   so the collapsed notification is critical, not warning.")

# 12 events in us-east, 5 of them critical -> the group must page as critical.
by_group = {g["group"]: g for g in grouped}
assert by_group["region=us-east"]["severity"] == "critical", (
    f"grouping downgraded a critical alert to "
    f"{by_group['region=us-east']['severity']} — someone does not get paged")
assert by_group["region=us-west"]["severity"] == "warning", (
    "us-west has no critical alerts, so it must not be escalated")
assert sum(g["alert_count"] for g in grouped) == len(events), "grouping dropped alerts"


## 📝 Recording Alert Events

Every time an alert fires or resolves, we record it in the database. This creates an **audit trail** for post-incident reviews.

In [ ]:
# Record alert events in Postgres and query history

conn = get_db_connection()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Simulate some alert events
import random

# Make this cell idempotent — clear previous demo events so re-runs stay clean.
cursor.execute("TRUNCATE TABLE alert_events RESTART IDENTITY")

events_to_insert = [
    (1, "firing", 92.5),   # High CPU fired
    (3, "firing", 7.3),    # High Error Rate fired
    (1, "resolved", 45.2), # High CPU resolved
    (2, "firing", 88.1),   # High Memory fired
    (3, "resolved", 2.1),  # High Error Rate resolved
    (5, "firing", 3.5),    # Slow Responses fired
    (2, "resolved", 62.0), # High Memory resolved
    (5, "resolved", 0.8),  # Slow Responses resolved
]

for rule_id, state, value in events_to_insert:
    if state == "firing":
        cursor.execute(
            "INSERT INTO alert_events (alert_rule_id, state, value) VALUES (%s, %s, %s)",
            (rule_id, state, value)
        )
    else:
        # Mark the most recent firing event as resolved
        cursor.execute("""
            UPDATE alert_events
            SET resolved_at = NOW(), state = 'resolved'
            WHERE alert_rule_id = %s AND state = 'firing' AND resolved_at IS NULL
        """, (rule_id,))

conn.commit()

# Query the alert history
cursor.execute("""
    SELECT
        ae.id,
        ar.name AS alert_name,
        ar.severity,
        ae.state,
        ae.value,
        ae.fired_at,
        ae.resolved_at
    FROM alert_events ae
    JOIN alert_rules ar ON ae.alert_rule_id = ar.id
    ORDER BY ae.fired_at DESC
""")
events = cursor.fetchall()

print("📜 Alert Event History")
print("=" * 85)
print(f"{'Alert':<22} {'Severity':<10} {'State':<10} {'Value':>7} {'Fired At':<20} {'Resolved At'}")
print("-" * 85)
for e in events:
    fired = str(e['fired_at'])[:19] if e['fired_at'] else ""
    resolved = str(e['resolved_at'])[:19] if e['resolved_at'] else "(still firing)"
    print(f"{e['alert_name']:<22} {e['severity']:<10} {e['state']:<10} {e['value']:>7.1f} {fired:<20} {resolved}")

print()
print(f"Total events: {len(events)}")
print()
print("💡 This history is essential for post-incident reviews.")
print("   'How long was the alert firing? What was the peak value?'")

conn.close()

## ⚡ Deep Dive: Reducing Alert Latency

Prometheus evaluates our rules every 15 seconds (`evaluation_interval` in `prometheus.yml`);
a hand-rolled evaluator like the one above commonly runs every 30-60 seconds. Either way, a
threshold breached one moment after an evaluation goes unnoticed until the next one — up to a
full interval of pure detection delay, before the `for:` clock even starts.

### Options for Faster Alerts

| Approach | Latency | Complexity | When to Use |
|----------|---------|------------|-------------|
| Poll every 60s | Up to 60s (30s average) | Low | Most alerts |
| Poll every 15s | Up to 15s (7.5s average) | Medium | Important SLOs |
| Stream processing (Flink) | 1-5s | High | Revenue-critical alerts |

### Stream Processing with Flink

```
Kafka (metrics) → Flink → Alert Events
                    │
                    ├── Reads metrics as they arrive (not on a schedule)
                    ├── Maintains rolling windows in memory
                    └── Fires alert immediately when window breaches threshold
```

For most organizations, **polling every 30-60 seconds is sufficient**.  
Stream-based alerting is worth the complexity only for critical-path metrics.


In [ ]:
# Simulate the latency difference between polling and streaming

import random
import math

random.seed(42)  # reproducible numbers in the table below


def simulate_detection_latency(polling_interval: int, num_incidents: int = 1000) -> list:
    """
    Simulate how long it takes to detect an incident with polling.
    The incident happens at a random time between evaluations.
    """
    latencies = []
    for _ in range(num_incidents):
        # Incident happens at random offset within the polling interval
        incident_offset = random.uniform(0, polling_interval)
        detection_latency = polling_interval - incident_offset
        latencies.append(detection_latency)
    return latencies


def percentile(values: list, q: float) -> float:
    """Nearest-rank percentile. sorted[int(q * n)] is off by one: for n=1000
    and q=0.99 it returns the 991st value, which is the 99.1st percentile."""
    s = sorted(values)
    return s[max(0, math.ceil(q * len(s)) - 1)]

print("⏱️ Alert Detection Latency Comparison")
print("=" * 60)

for interval in [60, 30, 15, 5]:
    latencies = simulate_detection_latency(interval)
    avg_lat = sum(latencies) / len(latencies)
    max_lat = max(latencies)
    p99_lat = percentile(latencies, 0.99)
    print(f"  Poll every {interval:>2}s:  avg={avg_lat:>5.1f}s  p99={p99_lat:>5.1f}s  "
          f"max={max_lat:>5.1f}s  (theory: avg={interval / 2:.1f}s, max={interval}s)")

    # Uniform arrival within the interval => mean detection delay is exactly
    # half the interval. If this drifts, the simulation stopped modelling polling.
    assert abs(avg_lat - interval / 2) < interval * 0.05, (
        f"expected avg ≈ {interval / 2:.1f}s for a {interval}s poll, got {avg_lat:.1f}s")
    assert max_lat <= interval, f"detection cannot exceed one polling interval"

print("  Stream (Flink):  avg=  2.0s  p99=  4.0s  max=  5.0s   ← illustrative, not simulated")
print()
print("💡 Polling at 15s gives 7.5s average detection — good enough for most cases.")
print("   Flink gives ~2s average but adds significant operational complexity.")
print()
print("   Remember: detection latency is SEPARATE from the 'for' duration, and it")
print("   is ADDITIVE. A 5-minute 'for' clause dominates a 15s evaluation interval,")
print("   so buying a streaming pipeline to shave 7 seconds off a 5-minute alert is")
print("   a 2% improvement for a 10× jump in operational complexity.")


## 📚 Summary

### Key Takeaways

1. **Alert rules** = query + threshold + duration. The duration prevents false positives from brief spikes — but remember `for:` requires the expression true at *every* evaluation, not on average.
2. **Alert states** follow a lifecycle: inactive → pending → firing → resolved.
3. **Polling evaluation** (every 30-60s) is simple and battle-tested — it's how Prometheus works.
4. **Notification routing** handles dedup, grouping, silencing, and escalation to prevent alert fatigue. Collapsing alerts must inherit the **highest** severity in the group, never the alphabetically largest one.
5. **Stream processing** (Flink) can reduce detection latency to seconds but adds complexity.
6. **Alert events** are recorded for post-incident review and audit trails.

### Next Up

In **Notebook 3**, we'll build **dashboards and visualizations** in Grafana — designing panels, optimizing queries, and understanding how to present metrics effectively.
